In [1]:
# general imports
import numpy as np
import pandas as pd
import sys
import time
import warnings
import os
import logging
from pathlib import Path
import pickle
import torch
warnings.filterwarnings('ignore')
PYTENSOR_FLAGS=''
logging.getLogger('matplotlib.font_manager').disabled = True
os.chdir("M:/1confiProj")
# append the path of the python scripts
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"\\scripts")

os.chdir("M:\\1confiProj")
from scripts.experiments import Experiment
from scripts.modelClassGPU import ConfiModel

data_path = 'P:\\3026008.02\\MNLE' # path to the data

device is : cpu


In [2]:
exp_name = 'MNLE-base'
m_id = 3
sim_data_path = os.path.join(data_path, exp_name, 'model_' + str(m_id), 'sim_data')
#print(sim_dara_path)
targetname = exp_name + '_model_' + str(m_id)
folder = Path(sim_data_path)

tensors_dat = []
tensor_par = []
files = sorted(folder.glob(targetname + "*"))  # sorted for reproducibility

# n_rep = 100
# exp = Experiment(exp_name)
# model = ConfiModel(m_id, exp.exp_config)
# print('---------model defined---------')
# trialcond = model.getCondRep(n_rep)

for f in files:
    #print(f)
    with open(f, "rb") as handle:
        sim_dat, theta_all, prior = pickle.load(handle)  # assume tensor
        #theta_all = torch.cat((theta_all, trialcond), dim=1)
    tensors_dat.append(sim_dat[:,4:])
    tensor_par.append(torch.cat((theta_all, sim_dat[:,:4]), dim=1))

# Concatenate row-wise (dim=0)


In [3]:
all_dat = torch.cat(tensors_dat, dim=0)
all_theta = torch.cat(tensor_par, dim=0)


In [4]:
all_dat

tensor([[-10.6102,   1.5450,   0.4100,   1.0000],
        [-13.0345,   5.2448,   0.0000,   1.0000],
        [-10.0608,  14.0275,   0.9600,   1.0000],
        ...,
        [  0.8510,   4.2339,   0.9600,   1.0000],
        [  9.3983,   2.2761,   1.0000,   1.0000],
        [  6.7301,   6.3675,   1.0000,   0.0000]])

In [5]:
all_theta

tensor([[  0.9880,  11.2719,  19.3575,  ..., -10.0000,   1.0000,   1.0000],
        [  1.7059,   3.8502,   5.2919,  ..., -10.0000,   1.0000,   1.0000],
        [  4.0406,   5.5196,  15.0207,  ..., -10.0000,   1.0000,   1.0000],
        ...,
        [  0.4701,   4.7729,  12.1785,  ...,  10.0000,   2.0000,   0.0000],
        [  1.9712,   3.1204,   0.2874,  ...,  10.0000,   2.0000,   0.0000],
        [  1.6708,  18.4067,   3.1205,  ...,  10.0000,   2.0000,   0.0000]])

In [6]:
#save the agg_data
sim_agg_data_path = os.path.join(data_path, exp_name, 'model_' + str(m_id), 'agg_data')
os.makedirs(sim_agg_data_path, exist_ok=True)
with open(os.path.join(sim_agg_data_path, targetname + '_agg.pickle'), 'wb') as handle:
    pickle.dump((all_dat, all_theta, prior), handle, protocol=pickle.HIGHEST_PROTOCOL)